[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap12_acoes_cdi_ipca.ipynb)

# Capítulo 12: Renda variável: ações, ETFs e BDRs

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Você compara, com dados reais, o Ibovespa, o CDI e a inflação desde 2000, vê as quedas pelo caminho e testa o efeito de aportes mensais.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
%pip install -q yfinance
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. Baixar os dados
Ibovespa pelo Yahoo Finance (índice de pontos, sem dividendos) e CDI e IPCA pelo Banco Central.

In [ ]:
import yfinance as yf
INICIO = '2000-01-01'   # MUDE AQUI
ibov = yf.download('^BVSP', start=INICIO, interval='1mo', auto_adjust=True, progress=False)['Close'].squeeze()
ibov.index = ibov.index.to_period('M').to_timestamp()
cdi = sgs(4391, inicio='01/01/2000') / 100
ipca = sgs(433, inicio='01/01/2000') / 100
m = pd.concat({'ibov': ibov.pct_change(), 'cdi': cdi, 'ipca': ipca}, axis=1).dropna()
m = m[INICIO:]
print('Período:', m.index[0].strftime('%m/%Y'), 'a', m.index[-1].strftime('%m/%Y'))

## 2. R$ 10 mil em cada um

In [ ]:
acum = 10_000 * (1 + m).cumprod()
ax = acum.rename(columns={'ibov':'Ibovespa (sem dividendos)','cdi':'CDI','ipca':'Inflação (IPCA)'}).plot(lw=2, color=[CORES['laranja'], CORES['azul'], CORES['vermelho']], title='R$ 10 mil aplicados no início do período')
eixo_reais(ax); ax.set_xlabel(''); plt.show()
print(acum.iloc[-1].map(brl))

## 3. As quedas pelo caminho
O gráfico mostra quanto cada investimento estava abaixo do seu pico anterior.

In [ ]:
dd = acum / acum.cummax() - 1
ax = (dd[['ibov']] * 100).plot(color=CORES['laranja'], lw=1.5, title='Ibovespa: distância do pico anterior (%)', legend=False)
ax.set_xlabel(''); plt.show()
print('Maior queda do Ibovespa no período:', pct(dd['ibov'].min()), 'em', dd['ibov'].idxmin().strftime('%m/%Y'))

## 4. Aportes mensais
Quem investe todo mês compra em momentos bons e ruins. Veja o resultado de R$ 500 por mês.

In [ ]:
APORTE = 500   # MUDE AQUI
res = {}
for c in ('ibov', 'cdi'):
    s = 0
    for r_ in m[c]:
        s = (s + APORTE) * (1 + r_)
    res[c] = s
total = APORTE * len(m)
print('Total aportado:', brl(total))
print('Ibovespa (sem dividendos):', brl(res['ibov']))
print('CDI (antes do imposto):', brl(res['cdi']))

**Atenção:** o Ibovespa em pontos não inclui os dividendos, que somam alguns pontos por ano. Os resultados do passado não se repetem necessariamente. Leia o capítulo 12 antes de tirar conclusões.

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 12 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap12/

Achou um erro? Escreva para contato@traderbrasil.com.